In [1]:
import os

### Write ```Dockerfile```

In [2]:
%%writefile Dockerfile

FROM python:3.9
    
# update package manager
RUN apt-get update

# update pip
RUN pip install --upgrade pip

# copy requirements
COPY requirements.txt .

# install dependencies
RUN pip install -r requirements.txt

# copy script into container
COPY script.py .

# run script when image is run
ENTRYPOINT ["python3", "script.py"]

Writing Dockerfile


### Write ```requirements.txt``` to local drive

In [3]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0

pandas==1.2.4
catboost==1.0.4
scikit_learn==0.24.1

Writing requirements.txt


### Write ```script.py``` to local drive

In [4]:
%%writefile script.py

import os
import pandas as pd
import numpy as np
import sys
import catboost as cb
import sklearn.metrics as skm

# get metric
def get_metric_by_year_month(target, y_hat, str_eval_metric):
    if str_eval_metric == 'AUC':
        return skm.roc_auc_score(y_true=target, y_score=y_hat)
    elif str_eval_metric == 'PRAUC':
        return skm.average_precision_score(y_true=target, y_score=y_hat)
    elif str_eval_metric == 'Logloss':
        return skm.log_loss(y_true=target, y_pred=y_hat)
    elif str_eval_metric == 'F1':
        return skm.f1_score(y_true=target, y_pred=y_hat)
    else:
        pass

# get string of feature to drop
str_col = sys.argv[1]
print(f'Column to drop: {str_col}')

# for testing
if str_col == 'ColumnNameTempValue':
    str_col = 'bankruptcystatus__ln'

# constants
str_project = '20231010-gen-xii'
str_target = 'target'
str_model = '02_pricing_pd'

###############################################################################
# HYPERPARAMETERS
###############################################################################
# get df_hyperparameters
str_filename = 'df_hyperparameters.csv'
str_uri = f's3://{str_project}/{str_model}/02_model/02_model/12_step_function/{str_filename}'
df = pd.read_csv(str_uri)
# convert to dict
dict_hyperparameters = dict(zip(df['keys'], df['values']))

# get number of iterations
int_n_iterations = int(dict_hyperparameters['INT_N_ITERATIONS'])
print(f'Iterations: {int_n_iterations}')

# get filename for training
str_filename_train = dict_hyperparameters['STR_FILENAME_TRAIN']
print(f'Training filename: {str_filename_train}')

# get filename for valid
str_filename_valid = dict_hyperparameters['STR_FILENAME_VALID']
print(f'Valid filename: {str_filename_valid}')

# get proportion of iterations to use as early stopping rounds
flt_prop_early_stopping = float(dict_hyperparameters['PROP_EARLY_STOPPING'])
print(f'Proportion early stopping: {flt_prop_early_stopping}')

# get eval metric
str_eval_metric = dict_hyperparameters['STR_EVAL_METRIC']
print(f'Eval metric: {str_eval_metric}')

##################################################################################

# get list of features
print('Importing list_of_starting_features...')
str_filename = 'df_cols_in_model.csv'
str_uri = f's3://{str_project}/{str_model}/02_model/02_model/01_lambda_get_starting_feats/{str_filename}'
list_cols_model = list(pd.read_csv(str_uri)['feature'])

# rm str_col
print(f'Removing {str_col} from model...')
list_cols_model = [col for col in list_cols_model if col != str_col]
print(f'After dropping {str_col}, there will be {len(list_cols_model)} features in the model')
# append target
list_cols_import = list_cols_model + [str_target]

# get lr
print('Getting learning rate...')
str_filename = 'df_tuning.csv'
str_uri = f's3://{str_project}/{str_model}/02_model/02_model/03_lambda_concat_tuning/{str_filename}'
df_tuning = pd.read_csv(str_uri)
# get learning rate
flt_learning_rate = df_tuning['learning_rate'].iloc[0]
print(f'Learning rate: {flt_learning_rate}')

# read training data
print('Reading training data...')
str_uri = f's3://{str_project}/{str_model}/02_model/00_preprocessing/02_make_dfs/{str_filename_train}'
df = pd.read_parquet(str_uri, columns=list_cols_import)

# class weights
print('Getting class weights...')
# these are coming from the test data set in gen 12 v2
flt_desired_0_prop = 0.6974
flt_desired_1_prop = 0.3026
# get the current proportions
ser_prop = df[str_target].value_counts(normalize=True)
# get currnt percentages
flt_current_0_prop = ser_prop[0]
flt_current_1_prop = ser_prop[1]
# get the weights
flt_0_weight = flt_desired_0_prop / flt_current_0_prop
flt_1_weight = flt_desired_1_prop / flt_current_1_prop
list_class_weights = [flt_0_weight, flt_1_weight]

# get the non numeric feats
print('Getting list of non-numeric columns...')
list_cols_non_numeric = []
for col in list_cols_model:
    if df[col].dtype not in ['float64','int64']:
        list_cols_non_numeric.append(col)

# build model
print('Building model...')
# pool data
pool_train = cb.Pool(
    df[list_cols_model], 
    df[str_target], 
    cat_features=list_cols_non_numeric,
)
del df

# read validation data
str_uri = f's3://{str_project}/{str_model}/02_model/00_preprocessing/02_make_dfs/{str_filename_valid}'
df = pd.read_parquet(str_uri, columns=list_cols_import)

# pool data
pool_valid = cb.Pool(
    df[list_cols_model], 
    df[str_target], 
    cat_features=list_cols_non_numeric,
)
del df

# constraints
dict_monotone_constraints = {
    # better
    'fltgrossmonthly__income_sum': -1, # as income increases, prediction gets better
    'fltapproveddowntotal__app': -1,
    'fltdowncash__app': -1,
    'bookvalue__app': -1,
    'ENG-dealership_age': -1,
    'bookvalue__app': -1,
    'fltdowncash__app': -1,
    'fltapproveddowntotal__app': -1,
    # worse
    'fltgrossmonthly__income_count': 1, # as count of income increases, prediction gets worse
    'ENG-loan_to_value': 1,
    'ENG-payment_to_income': 1,
    'ENG-vehicle_age': 1,
    'fltadvance__app': 1,
    'bigmileage_odometer__app': 1,
    'amtfinanced__app': 1,
    'miles_odometer__app': 1,
    'pti__app': 1,
    'fltadvance__app': 1,
    'ENG-loan_to_value': 1,
    'amtfinanced__app': 1,
}
# ensure features are in list_cols_model
dict_monotone_constraints = {key: val for key, val in dict_monotone_constraints.items() if key in list_cols_model}

# init class
cls_model_inference = cb.CatBoostClassifier(
    task_type='CPU',
    nan_mode='Min',
    random_state=42,
    eval_metric=str_eval_metric,
    iterations=int_n_iterations,
    learning_rate=flt_learning_rate,
    class_weights=list_class_weights,
    monotone_constraints=dict_monotone_constraints,
)

# fit
cls_model_inference.fit(
    pool_train,
    eval_set=[pool_valid],
    verbose=100,
    use_best_model=True,
    early_stopping_rounds=int(round(int_n_iterations*flt_prop_early_stopping)), 
)
del pool_train
del pool_valid

################################################################################################
# GET TRAINING EVAL METRIC
################################################################################################
print('Getting training eval metric...')

# import data
str_uri = f's3://{str_project}/{str_model}/02_model/00_preprocessing/02_make_dfs/{str_filename_train}'
df = pd.read_parquet(str_uri, columns=list_cols_import)

# get predictions
if str_eval_metric in ['AUC','PRAUC','Logloss']:
    # probabilities
    df['y_hat'] = cls_model_inference.predict_proba(df[cls_model_inference.feature_names_])[:,1]
elif str_eval_metric in ['F1']:
    # class
    df['y_hat'] = cls_model_inference.predict(df[cls_model_inference.feature_names_])
else:
    pass

# get eval metric - train
if str_eval_metric == 'AUC':
    flt_eval_metric_train = skm.roc_auc_score(y_true=df[str_target], y_score=df['y_hat'])
elif str_eval_metric == 'PRAUC':
    flt_eval_metric_train = skm.average_precision_score(y_true=df[str_target], y_score=df['y_hat'])
elif str_eval_metric == 'Logloss':
    flt_eval_metric_train = skm.log_loss(y_true=df[str_target], y_pred=df['y_hat'])
elif str_eval_metric == 'F1':
    flt_eval_metric_train = skm.f1_score(y_true=df[str_target], y_pred=df['y_hat'])

# save memory
del df

################################################################################################
# GET VALIDATION EVAL METRIC (WEIGHTED)
################################################################################################
print('Getting validation eval metric...')

# import data
str_uri = f's3://{str_project}/{str_model}/02_model/00_preprocessing/02_make_dfs/{str_filename_valid}'
df = pd.read_parquet(str_uri, columns=list_cols_import)

# get predictions
if str_eval_metric in ['AUC','PRAUC','Logloss']:
    # probabilities
    df['y_hat'] = cls_model_inference.predict_proba(df[cls_model_inference.feature_names_])[:,1]
elif str_eval_metric in ['F1']:
    # class
    df['y_hat'] = cls_model_inference.predict(df[cls_model_inference.feature_names_])
else:
    pass

# get eval metric - valid
if str_eval_metric == 'AUC':
    flt_eval_metric_valid = skm.roc_auc_score(y_true=df[str_target], y_score=df['y_hat'])
elif str_eval_metric == 'PRAUC':
    flt_eval_metric_valid = skm.average_precision_score(y_true=df[str_target], y_score=df['y_hat'])
elif str_eval_metric == 'Logloss':
    flt_eval_metric_valid = skm.log_loss(y_true=df[str_target], y_pred=df['y_hat'])
elif str_eval_metric == 'F1':
    flt_eval_metric_valid = skm.f1_score(y_true=df[str_target], y_pred=df['y_hat'])

# save memory
del df

################################################################################################
# CREATE OUTPUT DATA FRAME
################################################################################################
print('Creating output data frame...')

flt_diff = abs(flt_eval_metric_train - flt_eval_metric_valid)
dict_row = {
    'feature': str_col,
    'learning_rate': flt_learning_rate,
    'flt_eval_metric_train': flt_eval_metric_train,
    'flt_eval_metric_valid': flt_eval_metric_valid,
    'diff': flt_diff,
    'best_iteration': cls_model_inference.get_best_iteration(),
}
df = pd.DataFrame(dict_row, index=[0])
# save
str_filename = f'df_output_{str_col}.csv'
str_uri = f's3://{str_project}/{str_model}/02_model/02_model/04_batch_sensitivity_analysis/models/{str_filename}'
df.to_csv(str_uri, index=False)

Writing script.py


### Build and push to ECR

In [5]:
%%sh

# name the image
image=genxii-pd-sensitivity

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

Sending build context to Docker daemon  53.76kB
Step 1/7 : FROM python:3.9
 ---> 7ef94ac333fa
Step 2/7 : RUN apt-get update
 ---> Using cache
 ---> 2b52f23b12d0
Step 3/7 : RUN pip install --upgrade pip
 ---> Using cache
 ---> 57232942c9a7
Step 4/7 : COPY requirements.txt .
 ---> Using cache
 ---> 125194a6ed4b
Step 5/7 : RUN pip install -r requirements.txt
 ---> Using cache
 ---> 497fd5794989
Step 6/7 : COPY script.py .
 ---> 2744c3ba7044
Step 7/7 : ENTRYPOINT ["python3", "script.py"]
 ---> Running in 954fd5dd16bd
Removing intermediate container 954fd5dd16bd
 ---> 9c4de75114ae
Successfully built 9c4de75114ae
Successfully tagged genxii-pd-sensitivity:latest


WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'genxii-pd-sensitivity' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-pd-sensitivity]
2105ffe0ab6c: Preparing
1ce91218c96d: Preparing
86853ec93899: Preparing
8e1550fc5f05: Preparing
5727cd43f714: Preparing
78ecb2a2f011: Preparing
84062ebc4cf5: Preparing
2180aea5f54b: Preparing
86388e04a96b: Preparing
893507f6057f: Preparing
2353f7120e0e: Preparing
51a9318e6edf: Preparing
2180aea5f54b: Waiting
c5bb35826823: Preparing
86388e04a96b: Waiting
51a9318e6edf: Waiting
c5bb35826823: Waiting
893507f6057f: Waiting
2353f7120e0e: Waiting
78ecb2a2f011: Waiting
84062ebc4cf5: Waiting
1ce91218c96d: Layer already exists
5727cd43f714: Layer already exists
86853ec93899: Layer already exists
8e1550fc5f05: Layer already exists
78ecb2a2f011: Layer already exists
2180aea5f54b: Layer already exists
84062ebc4cf5: Layer already exists
86388e04a96b: Layer already exists
893507f6057f: Layer already exists
2353f7120e0e: Layer already exists
c5bb35826823: Layer already exists
51a9318e6edf: Layer already 

### Clean-up

In [6]:
# rm files
for str_file in ['Dockerfile','requirements.txt','script.py']:
    try:
        os.remove(f'./{str_file}')
    except:
        pass